In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from mlforecast import MLForecast
from tinyconformal.series import ConformalQuantileTimeSeriesRegressor
from sklearn.linear_model import LinearRegression

In [2]:
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

In [3]:
# Financial parameters
selling_price = 320      # Unit selling price
unit_cost = 90         # Unit acquisition cost
annual_holding_rate = 0.14 # Annual capital holding rate (12% p.a.)

days_obsoletes = 180

# 1. Underage Cost (Cu): Lost margin per unfulfilled unit
c_u = selling_price - unit_cost

# 2. Overage Cost (Co): Holding cost + daily obsolescence rate
daily_obsolescence_cost = unit_cost / days_obsoletes
daily_holding_cost = (unit_cost * annual_holding_rate) / 365

estimated_holding_days = 30
c_o = (daily_holding_cost + daily_obsolescence_cost) * estimated_holding_days

test = test.copy()
test.loc[:,"cu"] = float(c_u)
test.loc[:,"co"] = float(c_o)

# RandomForestQuantileRegressor

In [4]:

class QuantileRF(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        quantile=0.5,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        **kwargs,
    ):
        self.quantile = quantile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.kwargs = kwargs

    def fit(self, X, y):
        self.model_ = RandomForestQuantileRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            **self.kwargs,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X, quantiles=self.quantile)

In [5]:
def model_callable():
    models = {
        "RF-lo-90": QuantileRF(
            quantile=0.05, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-hi-90": QuantileRF(
            quantile=0.95, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-50": QuantileRF(
            quantile=0.50, n_estimators=100, max_depth=8, random_state=42
        ),
        "LinearRegression": LinearRegression()
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )


models = model_callable()

cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     interval_pairs=[
         ("RF-lo-90", "RF-hi-90"),
     ],
     n_windows=7,
     alpha=0.10,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,interval_pairs,"[('RF-lo-90', ...)]"
,median_cols,None
,n_windows,7
,alpha,0.1
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [6]:
preds = cqr.predict_interval(h=12, X_df=test)

In [7]:
from typing import List, Union

def economic_loss(
    df: pd.DataFrame,
    models: List[str],
    id_col: str = "unique_id",
    target_col: str = "y",
    cost_understock: Union[str, float] = "cu",
    cost_overstock: Union[str, float] = "co",
) -> pd.DataFrame:
    """Calculate Total Economic Loss (financial cost in currency) for multiple models.

    Evaluates forecasting models based on the Newsvendor cost model, measuring
    the financial impact of stockouts (understocking) and excess inventory (overstocking)
    per SKU.

    Parameters
    ----------
    df : pd.DataFrame
        Evaluation DataFrame containing ground-truth values, model forecasts,
        and unit costs for understock and overstock.
    models : List[str]
        List of column names corresponding to the forecasting models to evaluate.
    id_col : str, default="unique_id"
        Column name identifying unique series or group identifiers.
    target_col : str, default="y"
        Column name containing actual target values.
    cost_understock : str or float, default="cu"
        Column name or fixed scalar value representing the unit cost of stockout.
    cost_overstock : str or float, default="co"
        Column name or fixed scalar value representing the unit cost of excess.

    Returns
    -------
    pd.DataFrame
        A DataFrame formatted with `id_col`, `metric` label ('economic_loss'), and
        columns for each evaluated model containing their respective financial losses.

    Notes
    -----
    Interpretation:
    - Measures financial penalty in monetary units (e.g., currency).
    - Calculated as: (Understock Units * Unit Cost of Understock) + (Overstock Units * Unit Cost of Overstock).
    - Always non-negative (>= 0). Lower values indicate higher business efficiency.

    Understock and Overstock Mechanics:
    - **Understock (Rupture):** Occurs when actual demand exceeds the model's forecast (y > forecast).
      Calculated as max(0, y - forecast), representing the units of unmet demand.
    - **Overstock (Excess):** Occurs when the model's forecast exceeds actual demand (forecast > y).
      Calculated as max(0, forecast - y), representing the unsold units left in inventory.

    Final Economic Loss Calculation:
    - For each SKU and period, the total monetary loss is obtained by weighting
      the shortage and excess units by their respective unit costs (C_u and C_o):
      Economic Loss = (Understock × C_u) + (Overstock × C_o)
    - Finally, these individual period losses are aggregated by summing them over time
      for each unique SKU (`id_col`).
    """
    if not models:
        raise ValueError("The 'models' list cannot be empty.")
    if not all(model in df.columns for model in models):
        missing_models = [model for model in models if model not in df.columns]
        raise ValueError(
            f"The following model columns are missing from the DataFrame: {missing_models}"
        )

    cu = (
        df[cost_understock]
        if isinstance(cost_understock, str) and cost_understock in df.columns
        else cost_understock
    )
    co = (
        df[cost_overstock]
        if isinstance(cost_overstock, str) and cost_overstock in df.columns
        else cost_overstock
    )

    understock = df[models].rsub(df[target_col], axis=0).clip(lower=0)
    overstock = df[models].sub(df[target_col], axis=0).clip(lower=0)
    loss_per_row = (understock.mul(cu, axis=0)) + (overstock.mul(co, axis=0))
    res = loss_per_row.groupby(df[id_col], observed=True).sum().reset_index()

    res.insert(1, "metric", "economic_loss")
    return res

In [8]:
preds.loc[:,"cu"] = float(c_u)
preds.loc[:,"co"] = float(c_o)

In [ ]:
from typing import Any, Dict, Tuple, Union
import numpy as np
import pandas as pd


def _compute_critical_quantile(cu: np.ndarray, co: np.ndarray) -> np.ndarray:
    denom = cu + co
    return np.where(denom == 0, 0.5, cu / denom)


def _extract_cost_array(
    df: pd.DataFrame,
    cost_input: Union[str, float, int, Dict[Union[str, Tuple[str, Any]], float]],
    id_col: str,
    time_col: str,
) -> np.ndarray:
    if isinstance(cost_input, str):
        return df[cost_input].to_numpy(dtype=float)
    elif isinstance(cost_input, (int, float)):
        return np.full(len(df), float(cost_input), dtype=float)
    elif isinstance(cost_input, dict):
        if not cost_input:
            raise ValueError("Cost dictionary cannot be empty.")
            
        first_key = next(iter(cost_input.keys()))
        
        if isinstance(first_key, tuple):
            idx = pd.MultiIndex.from_frame(df[[id_col, time_col]])
            s_map = pd.Series(cost_input)
            return idx.map(s_map).to_numpy(dtype=float)
        else:
            return df[id_col].map(cost_input).to_numpy(dtype=float)
    else:
        raise TypeError(
            f"Cost input must be a column name (str), numeric scalar (float/int), "
            f"or dict mapping IDs or (ID, Time) tuples to values. Received: {type(cost_input)}"
        )


def _interpolate_linear(
    q_star: np.ndarray,
    q_lo: np.ndarray,
    q_hi: np.ndarray,
    p_lo: float,
    p_hi: float,
    q_med: np.ndarray | None = None,
    p_med: float = 0.50,
) -> np.ndarray:
    """Unified vectorized 2-point or 3-point (2-segment) piecewise linear interpolation."""
    
    if q_med is None:
        t = np.clip((q_star - p_lo) / (p_hi - p_lo), 0.0, 1.0)
        return q_lo + t * (q_hi - q_lo)

    t_lo = np.clip((q_star - p_lo) / (p_med - p_lo), 0.0, 1.0)
    res_lo = q_lo + t_lo * (q_med - q_lo)

    t_hi = np.clip((q_star - p_med) / (p_hi - p_med), 0.0, 1.0)
    res_hi = q_med + t_hi * (q_hi - q_med)

    return np.where(q_star <= p_med, res_lo, res_hi)


# -------------------------------------------------------------------------
# Public API Class with Detailed English Docstrings
# -------------------------------------------------------------------------

class NewsvendorSolver:
    """Vectorized Newsvendor Inventory Model Solver for Panel Data.

    This class solves the classic inventory/capacity decision problem under
    uncertainty (Newsvendor Problem) directly applied to probabilistic forecast 
    DataFrames in standard Nixtla ecosystem formats (e.g., StatsForecast, 
    NeuralForecast).

    General Workflow:
        1. **Critical Quantile Calculation (q_star):** Computes the optimal service 
           level q_star = c_u / (c_u + c_o) for each time series and period.
        2. **Probabilistic Mapping:** Maps prediction interval bounds (and optionally 
           the median) to cumulative probabilities.
        3. **Optimal Point Interpolation (y_star):** Estimates the target inventory 
           quantity that satisfies q_star using 2-point linear or 3-point 
           piecewise linear interpolation.
        4. **Adjustment and Physical Bounding:** Enforces non-negativity and clips 
           the final decision within bounds derived from the forecasts.

    Global Attention Points:
        - **Nixtla Compatibility:** The returned DataFrame preserves sorting by 
          `[id_col, time_col]`.
        - **Dictionary Performance:** When using dictionaries for c_u or c_o,
          lookup uses Pandas `MultiIndex` vectorization to prevent Python loops.
        - **CDF Monotonicity:** Crossed quantile errors from forecasting models 
          (e.g., q_lo > q_hi) are resolved internally via sequential clipping rules.

    Examples:
        >>> import pandas as pd
        >>> df_forecast = pd.DataFrame({
        ...     "unique_id": ["A", "A"],
        ...     "ds": ["2026-01-01", "2026-01-02"],
        ...     "lo-90": [10.0, 15.0],
        ...     "hi-90": [50.0, 60.0],
        ...     "median": [25.0, 30.0]
        ... })
        >>> solver = NewsvendorSolver()
        >>> res = solver.optimize(
        ...     df=df_forecast,
        ...     interval_pair=("lo-90", "hi-90"),
        ...     cu=10.0,
        ...     co=2.0,
        ...     median_col="median"
        ... )
    """

    @staticmethod
    def optimize(
        df: pd.DataFrame,
        interval_pair: Tuple[str, str],
        cu: Union[str, float, Dict[Union[str, Tuple[str, Any]], float]],
        co: Union[str, float, Dict[Union[str, Tuple[str, Any]], float]],
        level: int = 90,
        median_col: str | None = None,
        id_col: str = "unique_id",
        time_col: str = "ds",
        output_col: str = "y_optimal",
    ) -> pd.DataFrame:
        """Executes inventory optimization based on underage and overage costs.

        Calculates the optimal order quantity y_star by aligning the critical ratio 
        q_star with the empirical demand distribution represented by the forecast 
        quantiles in the DataFrame.

        Args:
            df (pd.DataFrame): Input DataFrame containing probabilistic forecasts.
            interval_pair (Tuple[str, str]): Column name tuple `(lower_col, upper_col)` 
                representing the prediction interval bounds (e.g., `("lo-90", "hi-90")`).
            cu (Union[str, float, Dict]): Underage/shortage cost (c_u). Can be:
                - Scalar (`float`/`int`): Constant cost across the entire panel.
                - `str`: Column name in `df` containing variable costs per row.
                - `dict`: Cost mapping by ID `{id: cost}` or tuple `{(id, ds): cost}`.
            co (Union[str, float, Dict]): Overage/holding cost (c_o). Accepts the 
                same input formats as `cu`.
            level (int, optional): Prediction interval level as a percentage (e.g., 90 for 90%). 
                Determines cumulative probabilities p_lo = (100 - level) / 200 and 
                p_hi = 1 - p_lo. Defaults to 90.
            median_col (str | None, optional): Name of the median column (Quantile 0.50). 
                If provided, performs 3-point (2-segment) piecewise linear interpolation. 
                If `None`, falls back to 2-point linear interpolation. Defaults to `None`.
            id_col (str, optional): Identifier column for unique time series. Defaults to `"unique_id"`.
            time_col (str, optional): Timestamp/date column. Defaults to `"ds"`.
            output_col (str, optional): Name of the output column for optimized quantities 
                in the returned DataFrame. Defaults to `"y_optimal"`.

        Returns:
            pd.DataFrame: A copy of the DataFrame sorted by `[id_col, time_col]` with the 
            calculated optimal values in `output_col`.

        Raises:
            ValueError: If `level` is not strictly between `(0, 100)` or if cost dictionaries are empty.
            TypeError: If the cost input type for `cu` or `co` is unsupported.

        Intentional Clips and Truncations:
            1. **Physical Non-negativity (`q_lo >= 0`, `q_hi >= 0`, `q_med >= 0`):**
               Applies `np.maximum(0.0, ...)` to ensure negative probabilistic model predictions 
               do not leak into physical inventory decisions.
            2. **Monotonicity Enforcement (`q_lo <= q_med <= q_hi`):**
               Sequentially corrects inverted quantile boundaries:
               - `q_med = max(q_lo, q_med)`
               - `q_hi = max(q_med, q_hi)`
            3. **Linear Boundary Clipping (`y_final` in range `[q_lo, q_hi]`):**
               Applies `np.clip(y_final, q_lo, q_hi)` to ensure that even for extreme critical 
               quantiles (`q_star < p_lo` or `q_star > p_hi`), the final order quantity remains 
               bounded within the prediction interval bounds.

        Attention Points:
            - **Zero-Division Safety:** If `cu + co == 0` for any row, the critical quantile 
              defaults to `0.5` (median) to prevent zero-division runtime errors.
            - **Out-of-Bounds Quantiles (`q_star` outside `[p_lo, p_hi]`):** Critical quantiles 
              falling outside the prediction interval are capped at `p_lo` or `p_hi` 
              (returning `q_lo` or `q_hi`), maintaining conservative inventory decisions.
            - **Missing Keys in Cost Dicts:** Unmatched IDs or timestamps in cost dictionaries 
              will resolve to `NaN` in the optimal output column.
        """
        if not (0 < level < 100):
            raise ValueError("The 'level' parameter must be strictly between 0 and 100.")

        if id_col in df.columns and time_col in df.columns:
            df_res = df.sort_values([id_col, time_col]).copy()
        else:
            df_res = df.copy()

        lo_col, hi_col = interval_pair

        p_lo = (100.0 - level) / 200.0
        p_hi = 1.0 - p_lo

        cu_arr = _extract_cost_array(df_res, cu, id_col, time_col)
        co_arr = _extract_cost_array(df_res, co, id_col, time_col)
        q_star = _compute_critical_quantile(cu=cu_arr, co=co_arr)

        # Clip 1: Non-negative physical baseline constraint
        q_lo = np.maximum(0.0, df_res[lo_col].to_numpy(dtype=float))
        q_hi = np.maximum(0.0, df_res[hi_col].to_numpy(dtype=float))

        if median_col is not None:
            q_med = np.maximum(0.0, df_res[median_col].to_numpy(dtype=float))
            # Clip 2: Enforce non-decreasing monotonicity across 3 quantile points
            q_med = np.maximum(q_lo, q_med)
            q_hi = np.maximum(q_med, q_hi)
        else:
            q_med = None
            # Clip 2 (Simplified): Enforce monotonicity across 2 quantile points
            q_hi = np.maximum(q_lo, q_hi)

        y_final = _interpolate_linear(
            q_star=q_star,
            q_lo=q_lo,
            q_hi=q_hi,
            p_lo=p_lo,
            p_hi=p_hi,
            q_med=q_med,
            p_med=0.50,
        )

        # Clip 3: Final truncation to boundary bounds [q_lo, q_hi]
        df_res[output_col] = np.clip(y_final, q_lo, q_hi)
        return df_res

In [143]:
%%time
res = NewsvendorSolver.optimize(preds, interval_pair=("RF-lo-90-cqr", "RF-hi-90-cqr"), level=90, median_col="RF-50", co="co", cu="cu")
res.loc[:, "y"] = test["y"].values

CPU times: user 2.34 ms, sys: 840 μs, total: 3.18 ms
Wall time: 2.56 ms


In [144]:
economic_loss(res, ["RF-50", "RF-lo-90-cqr", "RF-hi-90-cqr", "LinearRegression", "y_optimal"], id_col="unique_id")

,unique_id,metric,RF-50,RF-lo-90-cqr,RF-hi-90-cqr,LinearRegression,y_optimal
0,1,economic_loss,277949.49863,629590.5,17159.713151,124410.902034,16182.700296
